In [1]:
import sys

sys.path.append("..")
import numpy as np
import torch
import torch._dynamo.config
import torch._inductor.config
from tqdm import trange

torch._inductor.config.coordinate_descent_tuning = True
torch._inductor.config.triton.unique_kernel_names = True
torch._inductor.config.fx_graph_cache = (
    True  # Experimental feature to reduce compilation times, will be on by default in future
)
from IPython.display import Video

from utils.gpt import GPT, GPTConfig
from utils.video import transpose_and_clip, write_video
from utils.vqvae import CompressorConfig, Decoder

In [2]:
# load model
config = GPTConfig()
with torch.device("meta"):
    model = GPT(config)
model.load_state_dict_from_url(
    "https://huggingface.co/commaai/commavq-gpt2m/resolve/main/pytorch_model.bin",
    assign=True,
)
model = model.eval().to(device="cuda", dtype=torch.bfloat16)

In [3]:
model

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(1025, 1024)
    (wpe): Embedding(2580, 1024)
    (h): ModuleList(
      (0-23): 24 x TransformerBlock(
        (attn): Attention(
          (c_attn): Linear(in_features=1024, out_features=3072, bias=True)
          (c_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (mlp): FeedForward(
          (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
          (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
        )
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=1025, bias=False)
)

In [4]:
config

GPTConfig(block_size=2580, vocab_size=1025, n_layer=24, n_head=16, dim=1024, intermediate_size=4096, tokens_per_frame=129)

In [5]:
# compile

# model.decode_one_token = torch.compile(
#     model.decode_one_token, mode="reduce-overhead", fullgraph=True
# )
idx = torch.randint(
    0, config.vocab_size, (config.block_size - config.tokens_per_frame,), device="cuda"
)
y = model.generate(idx, config.tokens_per_frame)

In [6]:
# load tokens
tokens_condition = np.load("../examples/tokens.npy").astype(np.int32)
tokens_condition = np.c_[
    np.ones(len(tokens_condition), dtype=np.int32) * config.bos_token, tokens_condition
]
tokens_condition = tokens_condition[
    -(config.block_size // config.tokens_per_frame - 1) :
].reshape(-1)
tokens_condition = torch.tensor(tokens_condition, device="cuda")

In [7]:
tokens_condition

tensor([1024,  547,  412,  ...,  134,  839,  853], device='cuda:0',
       dtype=torch.int32)

In [8]:
# generate! (slow...)
NEW_FRAMES = 20 * 5
for _ in trange(NEW_FRAMES):
    tokens = model.generate(
        tokens_condition[-(config.block_size - config.tokens_per_frame) :],
        config.tokens_per_frame,
    )
    tokens_condition = torch.cat([tokens_condition, tokens], axis=0)

100%|██████████| 100/100 [00:43<00:00,  2.31it/s]


In [9]:
# reshape and remove BOS token
tokens_condition = tokens_condition.reshape(-1, config.tokens_per_frame)
tokens_condition = tokens_condition[:, 1:].to(dtype=torch.int64)

In [10]:
# load model
config = CompressorConfig()
with torch.device("meta"):
    decoder = Decoder(config)
decoder.load_state_dict_from_url(
    "https://huggingface.co/commaai/commavq-gpt2m/resolve/main/decoder_pytorch_model.bin",
    assign=True,
)
decoder = decoder.eval().to(device="cuda")

In [11]:
# decode generated tokens to video (same as decode.ipynb)
decoded_video = []
with torch.no_grad():
    for i in trange(len(tokens_condition)):
        decoded = decoder(tokens_condition[i][None])
        decoded_video.append(decoded)
decoded_video = torch.cat(decoded_video, dim=0).cpu().numpy()

100%|██████████| 119/119 [00:00<00:00, 143.89it/s]


In [12]:
# transpose and format video
decoded_video = transpose_and_clip(decoded_video)

In [14]:
# save video
save_dst = "../examples/generated.mp4"
write_video(decoded_video, save_dst, fps=20)
Video(save_dst, embed=True, width=700)